# Fongbe ASR — KenLM sur le modèle v2

1. Entraîne un KenLM 4-grammes sur **toutes les transcriptions** (reliable + religious + uncertain)
2. Décode les 300 voix de validation avec `pyctcdecode` + LM
3. Compare le TSER officiel : **greedy vs KenLM**

Prérequis : runtime GPU recommandé (inférence rapide), CPU suffit sinon.

In [ ]:
# === 0. Setup ===
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
os.makedirs('/content/work', exist_ok=True)

# libs
!pip install -q "transformers==5.14.1" soundfile pyctcdecode "https://github.com/kpu/kenlm/archive/master.zip"
# binaires kenlm (lmplz / build_binary)
if not os.path.exists('/content/kenlm-src/build/bin/lmplz'):
    !git clone -q --depth 1 https://github.com/kpu/kenlm /content/kenlm-src
    !mkdir -p /content/kenlm-src/build && cd /content/kenlm-src/build && cmake .. -DCMAKE_BUILD_TYPE=Release && make -j$(nproc)
if os.path.lexists('/content/kenlm-bin'):
    os.remove('/content/kenlm-bin')
os.symlink('/content/kenlm-src/build/bin', '/content/kenlm-bin')
print('lmplz:', os.path.exists('/content/kenlm-bin/lmplz'))

In [ ]:
# === 1. Données : CSV + scripts depuis Drive, val depuis GitHub ===
import urllib.request, shutil
from pathlib import Path

DATA = Path('/content/fongbe'); DATA.mkdir(exist_ok=True)
SRC = Path('/content/drive/MyDrive/fongbe-challenge/data')
for f in ['Train_reliable.csv','Train_uncertain.csv','Train_religious.csv',
          'syllabify.py','normalize.py','score.py']:
    shutil.copy2(SRC / f, DATA / f)
    print('✓', f)
BASE = 'https://raw.githubusercontent.com/Appolinairee/fongbe-challenge/master'
for f in ['valid_reference.csv','valid_prediction.csv']:
    urllib.request.urlretrieve(f'{BASE}/val/{f}', DATA / f)
print('csv ok')

In [ ]:
# === 2. Corpus KenLM : toutes les transcriptions normalisées ===
import pandas as pd, sys
sys.path.insert(0, str(DATA))
from normalize import tonal_target

frames = [pd.read_csv(DATA / f) for f in
          ['Train_reliable.csv','Train_religious.csv','Train_uncertain.csv']]
all_df = pd.concat(frames, ignore_index=True)
print(len(all_df), 'transcriptions')

with open('/content/lm_corpus.txt', 'w', encoding='utf-8') as f:
    for t in all_df.transcription.fillna(''):
        s = tonal_target(t).strip()
        if s:
            f.write(s + '\n')
print('corpus écrit')

In [ ]:
# === 3. Entraîner le KenLM 4-grammes ===
import subprocess, os
os.makedirs('/content/lm', exist_ok=True)
with open('/content/lm_corpus.txt') as fin, open('/content/lm/lm.arpa','w') as fout:
    subprocess.run(['/content/kenlm-bin/lmplz','-o','4'], stdin=fin, stdout=fout, check=True)
subprocess.run(['/content/kenlm-bin/build_binary','-o','4','/content/lm/lm.arpa','/content/lm/lm.bin'], check=True)
print('LM OK:', os.path.getsize('/content/lm/lm.bin')/1e6, 'Mo')

In [ ]:
# === 4. Audio de validation : les 300 wav du split v2 ===
import json, subprocess
val = pd.read_csv(DATA / 'valid_reference.csv')
rel = pd.read_csv(DATA / 'Train_reliable.csv').set_index('id')
val['audio_path'] = val.id.map(rel.audio_path)
print(len(val), 'fichiers val')

if not Path('/content/audio/train_reliable').exists():
    listing = json.loads(urllib.request.urlopen(
        'https://storage.googleapis.com/storage/v1/b/fongbe/o?maxResults=200&fields=items(name)').read())
    tars = sorted(i['name'] for i in listing['items']
                  if i['name'].startswith('audio_train_reliable'))
    for name in tars:
        tp = f'/content/{name}'
        if not Path(tp).exists():
            print('download', name, flush=True)
            subprocess.run(['wget','-q','-O',tp,f'https://storage.googleapis.com/fongbe/{name}'], check=True)
        subprocess.run(['tar','-xf',tp,'-C','/content'], check=True)
        os.remove(tp)
    Path('/content/audio').rename('/content/fongbe/audio') if Path('/content/audio').exists() and not Path('/content/fongbe/audio').exists() else None
print('audio ok')

In [ ]:
# === 5. Modèle v2 + décodage greedy vs KenLM ===
import torch, soundfile as sf, numpy as np
from transformers import Wav2Vec2BertProcessor, Wav2Vec2BertForCTC
from pyctcdecode import build_ctcdecoder

# localise le modèle v2 sur Drive
MODEL_DIR = None
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if 'model.safetensors' in files and 'vocab.json' in files:
        MODEL_DIR = root; break
print('modèle:', MODEL_DIR)
assert MODEL_DIR

device = 'cuda' if torch.cuda.is_available() else 'cpu'
processor = Wav2Vec2BertProcessor.from_pretrained(MODEL_DIR)
model = Wav2Vec2BertForCTC.from_pretrained(MODEL_DIR).to(device).eval()

# labels pyctcdecode : ids → chars, '|' → ' ' (séparateur de mots)
import json as _json
vocab = _json.loads(open(MODEL_DIR + '/vocab.json').read())
id2tok = {i: t for t, i in vocab.items()}
n = max(id2tok) + 1
labels = [id2tok.get(i, ' ') for i in range(n)]
labels = [' ' if t == '|' else t for t in labels]
labels = [t if t not in ('[UNK]','[PAD]') else ' ' for t in labels]

decoder = build_ctcdecoder(labels, kenlm_model_path='/content/lm/lm.bin',
                           alpha=0.5, beta=1.5)
print('decoder prêt')

In [ ]:
# === 6. Inférence + score des deux décodages ===
import sys as _s
_s.path.insert(0, str(DATA))
import score as official

AUDIO = Path('/content/fongbe/audio')
greedy_hyps, lm_hyps, refs = [], [], []

@torch.no_grad()
def logits_for(path):
    x, sr = sf.read(str(path), dtype='float32')
    if x.ndim > 1: x = x.mean(axis=1)
    f = processor(x, sampling_rate=16000, return_tensors='pt')
    return model(input_features=f.input_features.to(device),
                 attention_mask=f.attention_mask.to(device)).logits[0].float().cpu().numpy()

for k, row in enumerate(val.itertuples()):
    wav = AUDIO / row.audio_path.replace('audio/', '', 1)
    lg = logits_for(wav)
    greedy_hyps.append(processor.batch_decode(lg.argmax(-1)[None].numpy(), skip_special_tokens=True)[0])
    lm_hyps.append(decoder.decode(lg))
    refs.append(row.transcription)
    if (k+1) % 25 == 0: print(k+1, '/300', flush=True)

pairs_g = list(zip(refs, greedy_hyps))
pairs_l = list(zip(refs, lm_hyps))
mg = official.measure(pairs_g); ml = official.measure(pairs_l)
print('GREEDY :', {k: round(v,4) for k,v in mg.items() if k in ('score','TSER','TER_conditional')})
print('KENLM  :', {k: round(v,4) for k,v in ml.items() if k in ('score','TSER','TER_conditional')})

In [ ]:
# === 7. Résultats + sauvegarde Drive ===
res = pd.DataFrame({'id': val.id, 'ref': refs, 'greedy': greedy_hyps, 'kenlm': lm_hyps})
res.to_csv('/content/kenlm_val_results.csv', index=False)
DEST = '/content/drive/MyDrive/fongbe-kenlm'
os.makedirs(DEST, exist_ok=True)
subprocess.run(['cp','/content/lm/lm.bin', DEST + '/lm.bin'], check=False)
subprocess.run(['cp','/content/lm/lm.arpa', DEST + '/lm.arpa'], check=False)
subprocess.run(['cp','/content/kenlm_val_results.csv', DEST + '/kenlm_val_results.csv'], check=False)
print('sauvegardé dans Drive/fongbe-kenlm')